## STEP 5 - TERM BASED RETRIEVAL USING BM25

#### Adding a retrival mode setting i.e. which will decide which retrieval mode we will use bm25 or embedding

In [52]:
import sys
from pathlib import Path
PROJECT_ROOT = Path(".." ).resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

## helper method to reload specific file
import importlib
import config
importlib.reload(config)

<module 'config' from '/Users/hirakhan/Developer/AI-ML/rag-chatbot/config.py'>

In [53]:
from pathlib import Path
import numpy as np
import gradio as gr
from openai import OpenAI
from config import OPENAI_API_KEY, MODEL_NAME, EMBEDDING_MODEL, RETRIEVAL_MODE_BM25, RETRIEVAL_MODE_VECTOR
import chromadb

# 1. Initialize client & resolve paths
client = OpenAI(api_key=OPENAI_API_KEY)
client_chroma= chromadb.HttpClient(
    host="localhost",
    port=8000,
)

# vector or bm25
RETRIEVAL_MODE= 'vector' 

In [54]:
# 2. Load document & split into paragraphs
document_path = PROJECT_ROOT / "data" / "profile.txt"
document_text = document_path.read_text(encoding="utf-8")
paragraphs = [p.strip() for p in document_text.split("\n\n") if p.strip()]
documents = [{"id": i, "text": p} for i, p in enumerate(paragraphs)]

In [55]:
def retrieve(query):
    if RETRIEVAL_MODE == RETRIEVAL_MODE_VECTOR:
        return retrieve_vector(query)

    elif RETRIEVAL_MODE == RETRIEVAL_MODE_BM25:
        return retrieve_bm25(query)

    else:
        raise ValueError(f"Unknown retrieval mode: {RETRIEVAL_MODE}")

In [56]:
# 3. Compute embeddings for all document paragraphs
embedded_documents = []
for doc in documents:
    response = client.embeddings.create(
        model=EMBEDDING_MODEL,
        input=doc["text"]
    )
    embedded_documents.append({
        "id": doc["id"],
        "text": doc["text"],
        "embedding": response.data[0].embedding
    })

In [57]:
# 4. Create/get collection from DB
collection= client_chroma.get_or_create_collection(
    name="john_doe_profile"
)

# 5. Adding the embedded data in the DB
collection.add(
    ids=[str(doc["id"]) for doc in embedded_documents],
    documents=[doc["text"] for doc in embedded_documents],
    embeddings=[doc["embedding"] for doc in embedded_documents],
)

In [58]:
def get_embedding(text):
    response = client.embeddings.create(
        model=EMBEDDING_MODEL,
        input=text
    )

    return response.data[0].embedding


def retrieve_vector(query):
    # Generate query embedding
    query_embedding = get_embedding(query)

    # Query Chroma
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=3,
        include=["documents", "metadatas", "distances"]
    )

    return results


def retrieve_bm25(query):
    # BM25 retrieval will be implemented here
    query_tokens = tokenize(query)
    scores = []

    for document_id in range(len(tokenized_documents)):

        score = calculate_bm25_score(
            query_tokens=query_tokens,
            document_id=document_id,
            tokenized_documents=tokenized_documents,
            document_stats=document_stats,
            document_frequency=document_frequency
        )

        scores.append((document_id, score))

    # Highest score first
    scores.sort(key=lambda x: x[1], reverse=True)

    return scores[:top_k]

In [59]:
def rag_chatbot(message, history):

    # Retrieve relevant chunks
    results = retrieve(message)

    # Get retrieved documents
    retrieved_chunks = results["documents"][0]

    # Combine chunks into context
    retrieved_context = "\n\n".join(retrieved_chunks)

    # Build RAG system prompt
    system_prompt = f"""
You answer questions using ONLY the retrieved context below.

If the answer is not present in the context, say:
"I could not find that information in the document."

Retrieved Context:
{retrieved_context}
"""

    messages = [
        {"role": "system", "content": system_prompt}
    ]

    # Add conversation history
    for user_msg, bot_msg in history:
        messages.append({
            "role": "user",
            "content": user_msg
        })
        messages.append({
            "role": "assistant",
            "content": bot_msg
        })

    # Add current question
    messages.append({
        "role": "user",
        "content": message
    })

    # Generate answer
    chat_response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=messages
    )

    return chat_response.choices[0].message.content

In [60]:
# 7. Launch Gradio ChatInterface
demo = gr.ChatInterface(
    fn=rag_chatbot,
    title="Step 3: TERM BASED RETRIEVAL - BM25",
    description="Ask questions about John Doe's profile."
)

In [ ]:
demo.launch(prevent_thread_lock=True)

In [ ]:
demo.close()

In [61]:
import re

def tokenize(text):
      # Lowercase
    text = text.lower()

    # Remove punctuation
    text = re.sub(r"[^\w\s]", "", text)

    # Split into terms
    tokens = text.split()

    # Remove empty tokens
    tokens = [token for token in tokens if token]

    return tokens

In [62]:
text = "John Doe worked at OpenAI. He developed AI systems!"

tokenize(text)

['john', 'doe', 'worked', 'at', 'openai', 'he', 'developed', 'ai', 'systems']

In [63]:
tokenized_documents = [
    tokenize(doc["text"])
    for doc in documents
]

In [64]:
print(type(documents))
print(type(documents[0]))
print(documents[0])
# tokenized_documents

<class 'list'>
<class 'dict'>
{'id': 0, 'text': 'Fictional Biography of John Doe\nJohn Doe: A Life of Curiosity, Innovation, and Service'}


In [65]:
for doc in documents:
    doc["tokens"] = tokenize(doc["text"])

In [66]:
from collections import Counter

# Frequency of terms in each document
term_frequencies = []

# Length of each document
document_lengths = []

for tokens in tokenized_documents:
    term_freq = Counter(tokens)

    term_frequencies.append(term_freq)
    document_lengths.append(len(tokens))


# Average document length
average_document_length = (
    sum(document_lengths) / len(document_lengths)
)


# Number of documents containing each term
document_frequency = Counter()

for tokens in tokenized_documents:
    unique_terms = set(tokens)

    for term in unique_terms:
        document_frequency[term] += 1

In [67]:
from collections import defaultdict

inverted_index = defaultdict(dict)

for doc_id, term_freq in enumerate(term_frequencies):

    for term, frequency in term_freq.items():

        inverted_index[term][doc_id] = frequency

In [68]:
inverted_index["john"]

{0: 2,
 1: 2,
 2: 2,
 3: 1,
 4: 1,
 5: 1,
 6: 1,
 7: 1,
 8: 1,
 9: 1,
 10: 1,
 11: 1,
 12: 1,
 14: 1,
 15: 1,
 16: 1,
 17: 1,
 18: 2,
 19: 1,
 20: 1}

In [69]:
import math

def calculate_idf(term, document_count, document_frequency):
    return math.log(
        1 + (document_count - document_frequency + 0.5)
        / (document_frequency + 0.5)
    )


In [83]:
document_stats = {
    "lengths": {},
    "term_frequencies": {},
    "average_length": 1
}

def calculate_bm25_score(
    query_tokens,
    document_id,
    tokenized_documents,
    document_stats,
    document_frequency,
    k1=1.5,
    b=0.75
):
    
    total_documents = len(tokenized_documents)

    # Get tokens for this document
    document_tokens = tokenized_documents[document_id]

    # Document length
    document_length = len(document_tokens)

    # Average document length
    average_document_length = document_stats["average_length"]

    # Count term frequency
    term_frequencies = {}

    for token in document_tokens:
        term_frequencies[token] = term_frequencies.get(token, 0) + 1

    score = 0.0

    for term in query_tokens:

        # How many times does this term appear in this document?
        tf = term_frequencies.get(term, 0)

        # If term doesn't occur, it contributes nothing
        if tf == 0:
            continue

        # How many documents contain this term?
        df = document_frequency.get(term, 0)

        # Calculate IDF
        idf = calculate_idf(
            term,
            total_documents,
            df
        )

        # BM25 term-frequency component
        numerator = tf * (k1 + 1)

        denominator = (
            tf
            + k1 * (
                1 - b
                + b * (document_length / average_document_length)
            )
        )

        score += idf * (numerator / denominator)

    return score

In [72]:
query = "John innovation"
query_tokens = tokenize(query)

print(query_tokens)

['john', 'innovation']


In [73]:
if average_document_length == 0:
    average_document_length = 1

In [77]:
score = calculate_bm25_score(
    query_tokens=query_tokens,
    document_id=0,
    tokenized_documents=tokenized_documents,
    document_stats=document_stats,
    document_frequency=document_frequency
)

print(score)

0.3557145749297819


In [92]:
def retrieve_bm25(
    query,
    tokenized_documents,
    document_stats,
    document_frequency,
    inverted_index,
    top_k=3,
    k1=1.5,
    b=0.75
):
    # 1. Tokenize the query
    query_tokens = tokenize(query)

    # 2. Find candidate documents using the inverted index
    candidate_documents = set()

    for token in query_tokens:
        if token in inverted_index:
            candidate_documents.update(inverted_index[token])

    # 3. Calculate BM25 score for each candidate
    scored_documents = []

    for document_id in candidate_documents:
        score = calculate_bm25_score(
            query_tokens=query_tokens,
            document_id=document_id,
            tokenized_documents=tokenized_documents,
            document_stats=document_stats,
            document_frequency=document_frequency,
            k1=k1,
            b=b
        )

        # 4. Don't return zero-score documents
        if score > 0:
            scored_documents.append({
                "document_id": document_id,
                "score": score
            })

    # 5. Sort highest score first
    scored_documents.sort(
        key=lambda x: x["score"],
        reverse=True
    )

    # 6. Return top K
    return scored_documents[:top_k]

In [ ]:
query = "John innovation"

results = retrieve_bm25(
    query=query,
    tokenized_documents=tokenized_documents,
    document_stats=document_stats,
    document_frequency=document_frequency,
    inverted_index=inverted_index,
    top_k=3
)

results

[{'document_id': 0, 'score': 3.520313224743329},
 {'document_id': 21, 'score': 2.404944371598789},
 {'document_id': 18, 'score': 0.1603235217644447}]

## 6. Retrieve the highest-scoring documents

For a user query:
1. Tokenize the query
2. Use the inverted index to find candidate documents
3. Calculate BM25 scores
4. Sort by score (highest first)
5. Return the top 3 (skip zero-score documents)

In [95]:
# Wire document_stats to the values computed earlier
document_stats["average_length"] = average_document_length if average_document_length > 0 else 1
document_stats["lengths"] = {i: length for i, length in enumerate(document_lengths)}
document_stats["term_frequencies"] = {i: tf for i, tf in enumerate(term_frequencies)}


def display_bm25_results(query, top_k=3):
    """Run BM25 retrieval and print the top-ranked passages."""
    results = retrieve_bm25(
        query=query,
        tokenized_documents=tokenized_documents,
        document_stats=document_stats,
        document_frequency=document_frequency,
        inverted_index=inverted_index,
        top_k=top_k,
    )

    print(f"Query: '{query}'")
    print("=" * 90)
    print(f"{'Rank':<6} | {'Doc ID':<8} | {'BM25 Score':<12} | {'Text Snippet':<55}")
    print("-" * 90)

    for rank, hit in enumerate(results, start=1):
        doc_id = hit["document_id"]
        score = hit["score"]
        snippet = documents[doc_id]["text"].replace("\n", " ")[:52]
        if len(documents[doc_id]["text"]) > 52:
            snippet += "..."
        print(f"{rank:<6} | {doc_id:<8} | {score:<12.4f} | {snippet}")

    print("=" * 90 + "\n")
    return results


display_bm25_results("John innovation")
display_bm25_results("Where did John study?")

Query: 'John innovation'
Rank   | Doc ID   | BM25 Score   | Text Snippet                                           
------------------------------------------------------------------------------------------
1      | 0        | 3.5203       | Fictional Biography of John Doe John Doe: A Life of ...
2      | 21       | 2.4049       | John's fictional story illustrates how curiosity, co...
3      | 18       | 0.1603       | John also became increasingly interested in lifelong...

Query: 'Where did John study?'
Rank   | Doc ID   | BM25 Score   | Text Snippet                                           
------------------------------------------------------------------------------------------
1      | 5        | 2.7709       | After graduating with honors, John enrolled at the f...
2      | 0        | 0.2148       | Fictional Biography of John Doe John Doe: A Life of ...
3      | 18       | 0.1603       | John also became increasingly interested in lifelong...



[{'document_id': 5, 'score': 2.7708531602812485},
 {'document_id': 0, 'score': 0.21480862217282726},
 {'document_id': 18, 'score': 0.1603235217644447}]

## 7. Show term-level debugging information

For the top-ranked documents, show:
- Query terms
- Terms found in each document
- Term frequencies (TF)
- IDF values
- Final BM25 score

In [97]:
def calculate_bm25_breakdown(
    query_tokens,
    document_id,
    tokenized_documents,
    document_stats,
    document_frequency,
    k1=1.5,
    b=0.75,
):
    """Return total BM25 score plus per-term contribution details."""
    total_documents = len(tokenized_documents)
    document_tokens = tokenized_documents[document_id]
    document_length = len(document_tokens)
    average_document_length = document_stats["average_length"]

    term_frequencies = {}
    for token in document_tokens:
        term_frequencies[token] = term_frequencies.get(token, 0) + 1

    breakdown = []
    score = 0.0

    for term in query_tokens:
        tf = term_frequencies.get(term, 0)
        df = document_frequency.get(term, 0)
        idf = calculate_idf(term, total_documents, df) if df > 0 else 0.0

        if tf == 0:
            breakdown.append({
                "term": term,
                "found": False,
                "tf": 0,
                "df": df,
                "idf": round(idf, 4),
                "term_score": 0.0,
            })
            continue

        numerator = tf * (k1 + 1)
        denominator = tf + k1 * (1 - b + b * (document_length / average_document_length))
        term_score = idf * (numerator / denominator)
        score += term_score

        breakdown.append({
            "term": term,
            "found": True,
            "tf": tf,
            "df": df,
            "idf": round(idf, 4),
            "term_score": round(term_score, 4),
        })

    return round(score, 4), breakdown


def debug_bm25_query(query, top_k=3):
    query_tokens = tokenize(query)
    results = retrieve_bm25(
        query=query,
        tokenized_documents=tokenized_documents,
        document_stats=document_stats,
        document_frequency=document_frequency,
        inverted_index=inverted_index,
        top_k=top_k,
    )

    print(f"Query terms: {query_tokens}\n")

    for rank, hit in enumerate(results, start=1):
        doc_id = hit["document_id"]
        score, breakdown = calculate_bm25_breakdown(
            query_tokens=query_tokens,
            document_id=doc_id,
            tokenized_documents=tokenized_documents,
            document_stats=document_stats,
            document_frequency=document_frequency,
        )

        found_terms = [row["term"] for row in breakdown if row["found"]]
        snippet = documents[doc_id]["text"].replace("\n", " ")[:80]

        print(f"Rank {rank} | Doc {doc_id} | BM25 score: {score}")
        print(f"  Snippet: {snippet}...")
        print(f"  Terms found: {found_terms if found_terms else '(none)'}")
        print(f"  {'Term':<15} {'Found':<7} {'TF':<5} {'DF':<5} {'IDF':<8} {'Term Score':<12}")
        print(f"  {'-' * 58}")

        for row in breakdown:
            print(
                f"  {row['term']:<15} {str(row['found']):<7} {row['tf']:<5} "
                f"{row['df']:<5} {row['idf']:<8} {row['term_score']:<12}"
            )

        print()

    return results


debug_bm25_query("John innovation")

Query terms: ['john', 'innovation']

Rank 1 | Doc 0 | BM25 score: 3.5203
  Snippet: Fictional Biography of John Doe John Doe: A Life of Curiosity, Innovation, and S...
  Terms found: ['john', 'innovation']
  Term            Found   TF    DF    IDF      Term Score  
  ----------------------------------------------------------
  john            True    2     20    0.1151   0.2148      
  innovation      True    1     2     2.2192   3.3055      

Rank 2 | Doc 21 | BM25 score: 2.4049
  Snippet: John's fictional story illustrates how curiosity, continuous learning, compassio...
  Terms found: ['innovation']
  Term            Found   TF    DF    IDF      Term Score  
  ----------------------------------------------------------
  john            False   0     20    0.1151   0.0         
  innovation      True    1     2     2.2192   2.4049      

Rank 3 | Doc 18 | BM25 score: 0.1603
  Snippet: John also became increasingly interested in lifelong learning. He believed that ...
  Terms found: [

[{'document_id': 0, 'score': 3.520313224743329},
 {'document_id': 21, 'score': 2.404944371598789},
 {'document_id': 18, 'score': 0.1603235217644447}]

## 8. Test exact identifiers

Add passages containing employee IDs, order numbers, SKUs, error codes, person names, and acronyms.
Query using only those identifiers and compare BM25 vs vector retrieval.

In [98]:
identifier_passages = [
    "Employee record: John Doe's employee ID is EMP-78432. Contact HR for verification.",
    "Order confirmation: Your order number ORD-9921847 has shipped via standard delivery.",
    "Product catalog entry: SKU-AI-RAG-001 describes the retrieval-augmented generation starter kit.",
    "System log: ERROR-503-TIMEOUT occurred during the nightly batch sync on March 12.",
    "Conference speaker list includes Dr. Sarah Chen, keynote on responsible AI.",
    "Internal memo: The NLP team adopted RAG and LLM workflows for document search.",
]

# Extend the corpus with identifier-heavy passages
identifier_documents = [
    {"id": len(documents) + i, "text": text}
    for i, text in enumerate(identifier_passages)
]
extended_documents = documents + identifier_documents

extended_tokenized_documents = [tokenize(doc["text"]) for doc in extended_documents]
extended_term_frequencies = [Counter(tokens) for tokens in extended_tokenized_documents]
extended_document_lengths = [len(tokens) for tokens in extended_tokenized_documents]
extended_average_document_length = sum(extended_document_lengths) / len(extended_document_lengths)

extended_document_frequency = Counter()
for tokens in extended_tokenized_documents:
    for term in set(tokens):
        extended_document_frequency[term] += 1

extended_inverted_index = defaultdict(dict)
for doc_id, term_freq in enumerate(extended_term_frequencies):
    for term, frequency in term_freq.items():
        extended_inverted_index[term][doc_id] = frequency

extended_document_stats = {
    "lengths": {i: length for i, length in enumerate(extended_document_lengths)},
    "term_frequencies": {i: tf for i, tf in enumerate(extended_term_frequencies)},
    "average_length": extended_average_document_length if extended_average_document_length > 0 else 1,
}


def retrieve_bm25_extended(query, top_k=3):
    return retrieve_bm25(
        query=query,
        tokenized_documents=extended_tokenized_documents,
        document_stats=extended_document_stats,
        document_frequency=extended_document_frequency,
        inverted_index=extended_inverted_index,
        top_k=top_k,
    )


def retrieve_vector_inmemory(query, corpus_documents, top_k=3):
    """In-memory vector retrieval using cosine similarity."""
    query_embedding = np.array(get_embedding(query))
    scored = []

    for doc in corpus_documents:
        doc_embedding = np.array(get_embedding(doc["text"]))
        similarity = np.dot(query_embedding, doc_embedding) / (
            np.linalg.norm(query_embedding) * np.linalg.norm(doc_embedding)
        )
        scored.append({"document_id": doc["id"], "score": float(similarity)})

    scored.sort(key=lambda x: x["score"], reverse=True)
    return scored[:top_k]


def compare_bm25_vs_vector(query, corpus_documents, bm25_fn, top_k=3):
    bm25_results = bm25_fn(query, top_k=top_k)
    vector_results = retrieve_vector_inmemory(query, corpus_documents, top_k=top_k)

    print(f"Query: '{query}'")
    print("=" * 95)
    print(f"{'Method':<10} | {'Rank':<5} | {'Doc ID':<8} | {'Score':<12} | {'Snippet':<50}")
    print("-" * 95)

    for rank, hit in enumerate(bm25_results, start=1):
        doc_id = hit["document_id"]
        snippet = corpus_documents[doc_id]["text"][:48] + "..."
        print(f"{'BM25':<10} | {rank:<5} | {doc_id:<8} | {hit['score']:<12.4f} | {snippet}")

    for rank, hit in enumerate(vector_results, start=1):
        doc_id = hit["document_id"]
        snippet = corpus_documents[doc_id]["text"][:48] + "..."
        print(f"{'Vector':<10} | {rank:<5} | {doc_id:<8} | {hit['score']:<12.4f} | {snippet}")

    print("=" * 95 + "\n")


identifier_queries = [
    "EMP-78432",
    "ORD-9921847",
    "SKU-AI-RAG-001",
    "ERROR-503-TIMEOUT",
    "Sarah Chen",
    "RAG",
]

for q in identifier_queries:
    compare_bm25_vs_vector(q, extended_documents, retrieve_bm25_extended)

Query: 'EMP-78432'
Method     | Rank  | Doc ID   | Score        | Snippet                                           
-----------------------------------------------------------------------------------------------
BM25       | 1     | 22       | 4.3888       | Employee record: John Doe's employee ID is EMP-7...
Vector     | 1     | 22       | 0.4832       | Employee record: John Doe's employee ID is EMP-7...
Vector     | 2     | 23       | 0.2910       | Order confirmation: Your order number ORD-992184...
Vector     | 3     | 24       | 0.2673       | Product catalog entry: SKU-AI-RAG-001 describes ...

Query: 'ORD-9921847'
Method     | Rank  | Doc ID   | Score        | Snippet                                           
-----------------------------------------------------------------------------------------------
BM25       | 1     | 23       | 4.4576       | Order confirmation: Your order number ORD-992184...
Vector     | 1     | 23       | 0.4173       | Order confirmation: Your orde

## 9. Test semantic paraphrases

Ask questions using different vocabulary from the source text.
BM25 relies on lexical overlap; vector search should handle meaning without shared words.

In [99]:
paraphrase_tests = [
    {
        "query": "Where are you employed?",
        "target_hint": "work / career / company passages",
    },
    {
        "query": "What did John study at university?",
        "target_hint": "North Valley University / Computer Science",
    },
    {
        "query": "How does John feel about helping others learn?",
        "target_hint": "mentorship / teaching / workshops",
    },
]

for test in paraphrase_tests:
    query = test["query"]
    print(f"Expected topic: {test['target_hint']}")

    bm25_results = retrieve_bm25(
        query=query,
        tokenized_documents=tokenized_documents,
        document_stats=document_stats,
        document_frequency=document_frequency,
        inverted_index=inverted_index,
        top_k=3,
    )

    vector_results = retrieve_vector_inmemory(query, documents, top_k=3)

    print(f"Query: '{query}'")
    print(f"{'Method':<10} | {'Rank':<5} | {'Doc ID':<8} | {'Score':<12} | {'Snippet':<50}")
    print("-" * 90)

    for rank, hit in enumerate(bm25_results, start=1):
        doc_id = hit["document_id"]
        snippet = documents[doc_id]["text"][:48] + "..."
        print(f"{'BM25':<10} | {rank:<5} | {doc_id:<8} | {hit['score']:<12.4f} | {snippet}")

    for rank, hit in enumerate(vector_results, start=1):
        doc_id = hit["document_id"]
        snippet = documents[doc_id]["text"][:48] + "..."
        print(f"{'Vector':<10} | {rank:<5} | {doc_id:<8} | {hit['score']:<12.4f} | {snippet}")

    print("=" * 90 + "\n")

Expected topic: work / career / company passages
Query: 'Where are you employed?'
Method     | Rank  | Doc ID   | Score        | Snippet                                           
------------------------------------------------------------------------------------------
BM25       | 1     | 5        | 2.6588       | After graduating with honors, John enrolled at t...
Vector     | 1     | 8        | 0.3407       | Upon completing his degree in 2009, John accepte...
Vector     | 2     | 6        | 0.2121       | During his second year at university, John joine...
Vector     | 3     | 9        | 0.2080       | Over the next several years, John gained experti...

Expected topic: North Valley University / Computer Science
Query: 'What did John study at university?'
Method     | Rank  | Doc ID   | Score        | Snippet                                           
------------------------------------------------------------------------------------------
BM25       | 1     | 5        | 3.9819  

## 10. Compare manual implementation with a BM25 library

Install `rank_bm25` and compare scores, ranking, and tokenization behavior against our manual BM25.

In [88]:
%pip install rank_bm25 -q


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [89]:
from rank_bm25 import BM25Okapi


def rank_bm25_tokenize(text):
    """rank_bm25 default-style tokenization: lowercase + split on whitespace."""
    return text.lower().split()


def compare_manual_vs_library(query, top_k=5):
    query_tokens = tokenize(query)
    library_query_tokens = rank_bm25_tokenize(query)

    # Manual scores for all documents
    manual_scores = []
    for doc_id in range(len(tokenized_documents)):
        score = calculate_bm25_score(
            query_tokens=query_tokens,
            document_id=doc_id,
            tokenized_documents=tokenized_documents,
            document_stats=document_stats,
            document_frequency=document_frequency,
        )
        manual_scores.append((doc_id, score))

    manual_scores.sort(key=lambda x: x[1], reverse=True)
    manual_top = manual_scores[:top_k]

    # Library scores (uses its own tokenization on raw text)
    library_corpus = [rank_bm25_tokenize(doc["text"]) for doc in documents]
    bm25_lib = BM25Okapi(library_corpus)
    library_scores = bm25_lib.get_scores(library_query_tokens)

    library_ranked = sorted(
        [(doc_id, score) for doc_id, score in enumerate(library_scores)],
        key=lambda x: x[1],
        reverse=True,
    )
    library_top = library_ranked[:top_k]

    print(f"Query: '{query}'")
    print(f"Our tokenization:      {query_tokens}")
    print(f"Library tokenization:  {library_query_tokens}")
    print()

    print(f"{'Rank':<5} | {'Manual Doc':<12} | {'Manual Score':<14} | {'Library Doc':<13} | {'Library Score':<14} | {'Same Doc?':<10}")
    print("-" * 85)

    for rank in range(top_k):
        manual_doc, manual_score = manual_top[rank]
        library_doc, library_score = library_top[rank]
        same_doc = manual_doc == library_doc
        print(
            f"{rank + 1:<5} | {manual_doc:<12} | {manual_score:<14.4f} | "
            f"{library_doc:<13} | {library_score:<14.4f} | {str(same_doc):<10}"
        )

    print("-" * 85)

    manual_ids = [doc_id for doc_id, _ in manual_top]
    library_ids = [doc_id for doc_id, _ in library_top]
    print(f"Ranking match (top {top_k}): {manual_ids == library_ids}")

    # Tokenization difference demo
    sample = documents[0]["text"][:60]
    print(f"\nTokenization on sample text: '{sample}...'")
    print(f"  Our tokenize():          {tokenize(sample)}")
    print(f"  Library-style tokenize(): {rank_bm25_tokenize(sample)}")
    print()


compare_manual_vs_library("John innovation")
compare_manual_vs_library("North Valley University")
compare_manual_vs_library("EMP-78432")

Query: 'John innovation'
Our tokenization:      ['john', 'innovation']
Library tokenization:  ['john', 'innovation']

Rank  | Manual Doc   | Manual Score   | Library Doc   | Library Score  | Same Doc? 
-------------------------------------------------------------------------------------
1     | 0            | 3.5203         | 0             | 1.1383         | True      
2     | 21           | 2.4049         | 2             | 0.8104         | False     
3     | 18           | 0.1603         | 1             | 0.7663         | False     
4     | 2            | 0.1529         | 15            | 0.6608         | False     
5     | 1            | 0.1446         | 20            | 0.6608         | False     
-------------------------------------------------------------------------------------
Ranking match (top 5): False

Tokenization on sample text: 'Fictional Biography of John Doe
John Doe: A Life of Curiosit...'
  Our tokenize():          ['fictional', 'biography', 'of', 'john', 'doe', 'john'